In [22]:
from hmmlearn.hmm import GaussianHMM
import numpy as np
import joblib
import utils.data as d

HMM_FEATURES = ["es_rvol", "es_efficiency", "es_vol_ratio", "es_rel_volume"]
ZSCORE_WINDOW = 288

In [23]:
def zscore(df):
    out = df.copy()
    for col in HMM_FEATURES:
        mu = out[col].rolling(ZSCORE_WINDOW).mean()
        sigma = out[col].rolling(ZSCORE_WINDOW).std()
        out[f"{col}_z"] = (out[col] - mu) / sigma
    return out.dropna()

def train_hmm(df, n_restarts=10):
    X = df[[f"{c}_z" for c in HMM_FEATURES]].values
    best_model, best_score = None, -np.inf
    for i in range(n_restarts):
        model = GaussianHMM(
            n_components=2,
            covariance_type="full",
            n_iter=100,
            tol=1e-4,
            random_state=i,
        )
        model.fit(X)
        score = model.score(X)
        if score > best_score:
            best_model, best_score = model, score
    return best_model



In [24]:
data = d.get_data()
data = zscore(data)

model = train_hmm(data)
joblib.dump(model, "models/es_hmm.pkl")

Z_COLS = [f"{c}_z" for c in HMM_FEATURES]

means = data.groupby(model.predict(data[Z_COLS].values))[HMM_FEATURES].mean()
tradable_id = int(means["es_rvol"].idxmax())

posteriors = model.predict_proba(data[Z_COLS].values)
data["tradable_pct"] = posteriors[:, tradable_id]

print(f"LL: {model.score(data[Z_COLS].values):.2f}  |  tradable_id={tradable_id}\n")
print(data["tradable_pct"].describe())

LL: -26902.23  |  tradable_id=0

count    5.611000e+03
mean     3.148844e-01
std      4.485193e-01
min      1.875588e-55
25%      1.085238e-04
50%      7.739379e-04
75%      9.975883e-01
max      1.000000e+00
Name: tradable_pct, dtype: float64
